## actualizar retiro telef 

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [12]:
query = f"""
	select NUMERO_DOCUMENTO as dni_cliente,color_final from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_maestra = pd.read_sql(query, engine_kishin)

df_maestra["dni_cliente"] = (
    df_maestra["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_maestra.columns.tolist())


['dni_cliente', 'color_final']


In [4]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_formato = df_formato.merge(
    df_maestra,
    on='dni_cliente',
    how='left'
)

df_formato['fecha_visita']='2026-07-15'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [13]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [7]:
df_formato.shape

(2191, 20)

In [20]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [14]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'IQUITOS'}
{'TE'}


In [15]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'IQUITOS'}
{'TE'}


#### validar el codigo de agencia 

In [ ]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)

In [32]:
df_agencia[df_agencia['agencia_correo']=='SAN JUAN DE MIRAFLORES'].head()


,agencia_Formulario,agencia_base,agencia_base2,agencia_correo,correos
34,738224 - SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,yolinda.huayca@alfinbanco.pe


In [30]:
df_correo[df_correo['agencia_atencion']=='SAN JUAN DE MIRAFLORES'].head()


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
74,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,01236008,HERMENEGILDO CALSIN YUCRA,NaN,10000,952524186,SAN JUAN DE MIRAFLORES,2026-07-15,18:30:00,MANUAL
93,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,15627921,MARIZA OBDULIA ROSALES GIRIO DE CHINCHAY,VERDE OSCURO,9700,924119267,SAN JUAN DE MIRAFLORES,2026-07-15,12:00:00,MANUAL
132,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,16806886,VIDELMO YLATOMA BUSTAMANTE,NaN,8900,949886328,SAN JUAN DE MIRAFLORES,2026-07-15,14:45:00,MANUAL
192,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00205546,ARTURO CRUZ CAMPAÑA,NaN,6000,962095319,SAN JUAN DE MIRAFLORES,2026-07-15,14:30:00,MANUAL
216,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00214859,ENA ROSA BARRETO DIOSES,NaN,5200,922172667,SAN JUAN DE MIRAFLORES,2026-07-15,09:30:00,MANUAL


In [33]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

set()
{'VILLA MARIA 2', 'PC HUANCAYO', 'AREQUIPA PAMPILLA', 'CHIMBOTE', 'SAN JUAN DE MIRAFLORES'}


In [19]:
print(df_agencia["agencia_correo"].drop_duplicates().tolist())

['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']


In [ ]:
['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']MARIA 

In [ ]:
    {'VILLA MARIA 2', 'PC HUANCAYO', 'AREQUIPA PAMPILLA', 'CHIMBOTE', 'SAN JUAN DE MIRAFLORES'}

equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [34]:
print(df_correo["agencia_atencion"].drop_duplicates().tolist())


['HUANUCO', 'HUANCAYO', 'SAN MIGUEL', 'CUSCO LA CULTURA', 'TACNA', 'TRUJILLO CENTRO', 'TARAPOTO', 'EMANCIPACION', 'CHIMBOTE', 'MIRAFLORES', 'TRUJILLO AMERICA', 'HUACHO', 'COMAS', 'CHICLAYO BALTA', 'SAN JUAN DE LURIGANCHO', 'CASTILLA', 'AREQUIPA PAMPILLA', 'SULLANA', 'VENTANILLA', 'MOSHOQUEQUE', 'JESUS MARIA', 'PUCALLPA', 'ATE VITARTE', 'LOS OLIVOS', 'VILLA MARIA 2', 'SANTA ANITA', 'CAJAMARCA', 'AREQUIPA CAYMA', 'PISCO', 'HUARAZ', 'SAN JUAN DE MIRAFLORES', 'CHINCHA', 'HUARAL', 'ICA', 'SAN MARTIN', 'JULIACA 2', 'VILLA EL SALVADOR 2', 'TUMBES', 'CAÑETE', 'PUENTE PIEDRA', nan, 'TE']


In [ ]:

df_correo[df_correo['dni_cliente']=='09704310'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2190,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,NaN,18000,930162239,MIRAFLORES,2026-07-15,13:30:00,MANUAL


In [25]:
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,10019878,ANA MARIA ESTELA VILLANUEVA PEÑA,AMARILLO OSCURO,8000,944496036,HUANUCO,2026-07-15,11:45:00,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,10038527,ROXANA MARIBEL CHAVEZ HUAMAN,VERDE OSCURO,11600,913499652,HUANCAYO,2026-07-15,15:30:00,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,10019878,ANA MARIA ESTELA VILLANUEVA PEÑA,944496036,738224 - SAN JUAN DE MIRAFLORES,2026-07-15,8000,Derivacion
1,BOT,TARGET,10038527,ROXANA MARIBEL CHAVEZ HUAMAN,913499652,738252 - SANTA ANITA,2026-07-15,11600,Derivacion


In [26]:
df_correo['fecha_visita']='2026-07-15'
df_formulario['fecha_visita']='2026-07-15'


In [27]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

2191

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,09750804,AMADO VILLALTA,NaN,23000,942707381,CASTILLA,2026-07-08,14:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,08931784,LUIS GUILLERMO CHUQUIJAJAS,NaN,23000,950955962,VILLA EL SALVADOR 2,2026-07-08,14:30:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,09750804,AMADO VILLALTA,942707381,737490 - CASTILLA,2026-07-08,23000,Derivacion
1,00000001,TARGET,08931784,LUIS GUILLERMO CHUQUIJAJAS,950955962,732249 - VILLA EL SALVADOR 2,2026-07-08,23000,Derivacion
